# Predicting Risk of Dyslexia
This notebook contains the data cleaning, preprocessing, feature selection, and training pipeline for predicting the risk of dyslexia using cognitive game performance data.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
import os

# Verify data files exist
for file in ['Dyt-desktop.csv', 'Dyt-tablet.csv']:
    print(f"{file} exists:", os.path.exists(file))

## Load and Clean Data

In [ ]:
def cleanData(data):
    # Map Gender: Male -> 1, Female -> 2
    if 'Gender' in data.columns:
        data['Gender'] = data['Gender'].map({'Male': 1, 'Female': 2})
    # Map Dyslexia: No -> 0, Yes -> 1
    if 'Dyslexia' in data.columns:
        data['Dyslexia'] = data['Dyslexia'].map({'No': 0, 'Yes': 1})
    
    # Ensure all other columns are numeric
    for col in data.columns:
        if col not in ['Gender', 'Dyslexia']:
            data[col] = pd.to_numeric(data[col], errors='coerce')

# Read CSVs directly with the correct delimiter and NA values
desktopData = pd.read_csv('Dyt-desktop.csv', sep=';', na_values=['NULL', '', '(NA)'])
tabletData = pd.read_csv('Dyt-tablet.csv', sep=';', na_values=['NULL', '', '(NA)'])

cleanData(desktopData)
cleanData(tabletData)

print("Desktop data shape:", desktopData.shape)
print("Tablet data shape:", tabletData.shape)

## Handle Missing Values
Fill missing values in the tablet dataset using column averages.

In [ ]:
for col in tabletData.columns:
    if tabletData[col].isnull().any():
        mean_val = tabletData[col].mean()
        if not pd.isna(mean_val):
            tabletData[col] = tabletData[col].fillna(round(mean_val, 4))

## Drop Remaining Missing Columns
Drop any columns that are still missing values (meaning they were completely NULL in tabletData).

In [ ]:
cols_with_missing = [col for col in tabletData.columns if tabletData[col].isnull().any()]
print("Columns with missing values (completely NULL):", cols_with_missing)

reduced_desktopData = desktopData.drop(columns=cols_with_missing, errors='ignore')
reduced_tabletData = tabletData.drop(columns=cols_with_missing, errors='ignore')

## Select Features

In [ ]:
commonalityColumns = ['Gender', 'Age', 'Dyslexia']
for i in range(30):
    if (0 <= i < 12) or (13 <= i < 17) or i == 21 or i == 22 or i == 29:
        commonalityColumns.extend([
            f'Clicks{i+1}', f'Hits{i+1}', f'Misses{i+1}',
            f'Score{i+1}', f'Accuracy{i+1}', f'Missrate{i+1}'
        ])

# Keep only columns that exist in the reduced data
commonalityColumns = [col for col in commonalityColumns if col in reduced_desktopData.columns]

reduced_desktopData = reduced_desktopData[commonalityColumns]
reduced_tabletData = reduced_tabletData[commonalityColumns]

## Train RandomForestClassifier on Desktop Data

In [ ]:
y = reduced_desktopData['Dyslexia']
X = reduced_desktopData.drop(columns=['Dyslexia'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

rfc = RandomForestClassifier(random_state=42)
rfc.fit(X_train, y_train)
y_pred = rfc.predict(X_test)
print("Accuracy on desktop test set:", metrics.accuracy_score(y_test, y_pred))

## Evaluate on Tablet Data

In [ ]:
yTest = reduced_tabletData['Dyslexia']
XTest = reduced_tabletData.drop(columns=['Dyslexia'])

y_pred_tablet = rfc.predict(XTest)
print("Accuracy on tablet set:", metrics.accuracy_score(yTest, y_pred_tablet))